In [28]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
from datetime import datetime

In [ ]:
# CONFIG
dataset1k = "data/1.0k-28px-03-10-04:06.npz"
dataset5k = "data/5.0k-28px-03-10-04:13.npz"

DATA = dataset5k

In [30]:

print("Loading dataset...")
data = np.load(DATA)
X = data['X']
y = data['y']

print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Converting to tensors...")
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1) / 255.0
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1) / 255.0
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

batch_size = 64
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Ready! Training batches: {len(train_loader)}, Validation batches: {len(test_loader)}")

Loading dataset...
Splitting data...
Converting to tensors...
Ready! Training batches: 1663, Validation batches: 416


In [31]:
class SitelenPonaCNN(nn.Module):
    def __init__(self, num_classes=133):
        super(SitelenPonaCNN, self).__init__()
        
        # Layer 1: Sees the 1-channel grayscale image, outputs 32 feature maps
        # Padding=1 keeps the spatial dimensions at 28x28 before pooling
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        
        # Layer 2: Takes the 32 feature maps, outputs 64 more complex feature maps
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        
        # Max Pooling: Shrinks the image dimensions by half (2x2 window)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected Layers
        # After two pools, our 28x28 image shrinks to 14x14, then 7x7.
        # 64 channels * 7 height * 7 width = 3136 flat features
        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.dropout = nn.Dropout(0.5) # Randomly zeroes 50% of neurons to prevent overfitting
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Pass through Conv1 -> ReLU activation -> MaxPool
        x = self.pool(F.relu(self.conv1(x)))
        
        # Pass through Conv2 -> ReLU activation -> MaxPool
        x = self.pool(F.relu(self.conv2(x)))
        
        # Flatten the 3D tensor into a 1D vector for the linear layers
        x = torch.flatten(x, 1) 
        
        # Pass through fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x) # Output is raw scores (logits) for each of the 137 classes
        return x

In [32]:
# 1. Setup Device (Hardware Acceleration)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Initialize Model, Loss, and Optimizer
model = SitelenPonaCNN(num_classes=133).to(device)
criterion = nn.CrossEntropyLoss() # Perfect for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Training Loop Configuration
epochs = 10 # Start with 10 to see how fast it learns

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train() # Tell PyTorch we are training (enables dropout)
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        # Move data to GPU if available
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero out old gradients
        optimizer.zero_grad()
        
        # Forward pass: what does the model think this image is?
        outputs = model(inputs)
        
        # Calculate the error
        loss = criterion(outputs, labels)
        
        # Backward pass: calculate gradients
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        running_loss += loss.item()
        
    # --- VALIDATION PHASE ---
    model.eval() # Tell PyTorch we are evaluating (disables dropout)
    correct = 0
    total = 0
    val_loss = 0.0
    
    # We don't need to track gradients for validation, saving memory
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            
            # Track validation loss
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Get the highest scoring class prediction
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    # Print epoch stats
    train_loss_avg = running_loss / len(train_loader)
    val_loss_avg = val_loss / len(test_loader)
    val_accuracy = 100 * correct / total
    
    print(f"Epoch {epoch+1:02d}/{epochs} | "
          f"Train Loss: {train_loss_avg:.4f} | "
          f"Val Loss: {val_loss_avg:.4f} | "
          f"Val Accuracy: {val_accuracy:.2f}%")

# Save the trained weights so you don't have to retrain every time
dataname = DATA.split('/')[1].split('-')[0].replace('.0', '')
output_filename = f"models/{val_accuracy:.2f}%-{dataname}-{datetime.now().strftime("%m-%d-%H:%M")}.pth"
torch.save(model.state_dict(), output_filename)
print(f"Model saved to {output_filename}")

Training on device: mps
Epoch 01/10 | Train Loss: 2.2284 | Val Loss: 0.8197 | Val Accuracy: 78.49%
Epoch 02/10 | Train Loss: 1.0748 | Val Loss: 0.5332 | Val Accuracy: 85.29%
Epoch 03/10 | Train Loss: 0.8553 | Val Loss: 0.4103 | Val Accuracy: 88.59%
Epoch 04/10 | Train Loss: 0.7345 | Val Loss: 0.3433 | Val Accuracy: 90.26%
Epoch 05/10 | Train Loss: 0.6505 | Val Loss: 0.2984 | Val Accuracy: 91.50%
Epoch 06/10 | Train Loss: 0.5885 | Val Loss: 0.2690 | Val Accuracy: 92.24%
Epoch 07/10 | Train Loss: 0.5340 | Val Loss: 0.2392 | Val Accuracy: 93.00%
Epoch 08/10 | Train Loss: 0.4894 | Val Loss: 0.2193 | Val Accuracy: 93.41%
Epoch 09/10 | Train Loss: 0.4533 | Val Loss: 0.2095 | Val Accuracy: 93.85%
Epoch 10/10 | Train Loss: 0.4218 | Val Loss: 0.1868 | Val Accuracy: 94.24%
Model saved to models/94.24%-1k-03-10-04:43.pth
